# Detecting Water Leakage in Railway Toilets from Air-Pressure Cycle Time

**A classroom tutorial — simulated data + a simple classifier**

By the end of this notebook you will:

1. Understand why a leak changes the **air-pressure cycle time** in a vacuum toilet.
2. Simulate pressure-sensor data for healthy and leaky toilets.
3. Extract simple features (cycle period, duty cycle).
4. Fit a classifier (logistic regression + decision tree) and evaluate it.
5. Use the trained model to predict a brand-new recording.

## 1. The physics — why a leak shortens the cycle

Modern train toilets use a **vacuum system**:

- A small tank is held below atmospheric pressure by a **vacuum pump**.
- Tiny air leaks (through seals, valves, water traps) let outside air drift *into* the tank, so the pressure slowly rises.
- When pressure hits an **upper threshold** `p_high`, the pump turns **ON** and pulls pressure back down to a **lower threshold** `p_low`. Then it turns **OFF** again.
- The result is a periodic saw-tooth pressure signal.

If a water seal fails, more air leaks in per second. The pump now has to start much sooner — so the cycle period gets **shorter** and the pump's duty cycle gets **higher**.

**That is the signal we will train a classifier to detect.**

| Quantity | Healthy toilet | Leaky toilet |
|---|---|---|
| Cycle period (s) | long (~40–60) | short (~15–30) |
| Pump duty cycle | low (~15%) | high (~50%) |
| Pressure range | full swing p_low ↔ p_high | full swing p_low ↔ p_high |

## 2. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

RNG = np.random.default_rng(42)  # reproducible randomness

## 3. The simulator

We model the tank as a simple bang-bang controller:

- Between samples, pressure changes by `(leak_rate − pump_rate · pump_on) · dt`.
- Pump turns ON when pressure crosses `p_high`, OFF when it crosses `p_low`.
- A little Gaussian noise represents the real sensor.

`leak_rate` is the **knob** we change to switch between *healthy* and *leaky*.

In [ ]:
def simulate_pressure(duration_s=240,
                      sample_rate=10,
                      leak_rate=0.009,    # bar/s drift toward atmospheric
                      pump_rate=0.050,    # bar/s pulled down when pump ON
                      p_low=0.20,
                      p_high=0.55,
                      noise_std=0.005,
                      rng=None):
    """Return (time, pressure, pump_on) arrays for a vacuum tank."""
    rng = rng or np.random.default_rng()
    n = int(duration_s * sample_rate)
    dt = 1.0 / sample_rate

    p = np.empty(n)
    on = np.zeros(n, dtype=bool)
    p[0] = p_low + 0.5 * (p_high - p_low)
    on[0] = False

    for i in range(1, n):
        if on[i - 1]:
            new_p = p[i - 1] - pump_rate * dt + leak_rate * dt
            on[i] = new_p > p_low
        else:
            new_p = p[i - 1] + leak_rate * dt
            on[i] = new_p >= p_high
        p[i] = new_p + rng.normal(0.0, noise_std)

    t = np.arange(n) * dt
    return t, p, on

## 4. One healthy toilet vs one leaky toilet

Run the simulator twice with two different leak rates, then plot the raw pressure traces.

In [ ]:
t_h, p_h, on_h = simulate_pressure(leak_rate=0.009, rng=RNG)   # healthy
t_l, p_l, on_l = simulate_pressure(leak_rate=0.028, rng=RNG)   # leaky

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(t_h, p_h, lw=1)
axes[0].set_title('Healthy toilet — long, irregular cycles')
axes[0].set_ylabel('Tank pressure (bar)')
axes[0].grid(alpha=0.3)
axes[1].plot(t_l, p_l, lw=1, color='firebrick')
axes[1].set_title('Leaky toilet — short, frequent cycles')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Tank pressure (bar)')
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Turn a 4-minute trace into a feature vector

A classifier doesn't eat raw 2400-sample time series very well. We summarise each recording with four numbers:

1. **Mean cycle period** (s) — time between successive pump-ON events.
2. **Duty cycle** — fraction of samples with pump ON.
3. **Pressure range** — max minus min over the recording.
4. **Cycle period std** — variability of the period.

Of these, **mean cycle period** is the most physically meaningful — it falls as the leak rate rises.

In [ ]:
def extract_features(pressure, pump_on, sample_rate=10):
    dt = 1.0 / sample_rate
    # Rising edges of pump_on -> start of each new cycle
    edges = np.where(np.diff(pump_on.astype(int)) == 1)[0]
    if len(edges) >= 2:
        periods = np.diff(edges) * dt
        mean_period = periods.mean()
        std_period = periods.std()
    else:
        mean_period = len(pressure) * dt
        std_period = 0.0
    return {
        'mean_period_s':   mean_period,
        'duty_cycle':      pump_on.mean(),
        'pressure_range':  pressure.max() - pressure.min(),
        'period_std_s':    std_period,
    }

print('Healthy features:', extract_features(p_h, on_h))
print('Leaky   features:', extract_features(p_l, on_l))

## 6. Build a labelled dataset

We generate **120 healthy + 120 leaky** recordings, with leak rates drawn from two slightly *overlapping* ranges so the problem is non-trivial (some borderline samples will confuse the classifier — that is realistic).

In [ ]:
FEATURE_NAMES = ['mean_period_s', 'duty_cycle', 'pressure_range', 'period_std_s']

def make_dataset(n_per_class=120, rng=None):
    rng = rng or np.random.default_rng(0)
    X, y = [], []
    for _ in range(n_per_class):
        # healthy: small leak (slight overlap with leaky tail)
        lr = rng.uniform(0.007, 0.015)
        t, p, on = simulate_pressure(leak_rate=lr, rng=rng)
        X.append(list(extract_features(p, on).values()))
        y.append(0)
    for _ in range(n_per_class):
        # leaky: bigger leak (slight overlap with healthy tail)
        lr = rng.uniform(0.013, 0.030)
        t, p, on = simulate_pressure(leak_rate=lr, rng=rng)
        X.append(list(extract_features(p, on).values()))
        y.append(1)
    return np.array(X), np.array(y)

X, y = make_dataset(n_per_class=120, rng=np.random.default_rng(7))
print('X shape:', X.shape, ' label counts:', np.bincount(y))

## 7. Visualise the feature space

Plot the two most informative features against each other and colour by label. If the two classes form distinct clouds, even a linear classifier will work well.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X[y == 0, 0], X[y == 0, 1], label='healthy', alpha=0.7)
ax.scatter(X[y == 1, 0], X[y == 1, 1], label='leak',    alpha=0.7, color='firebrick')
ax.set_xlabel('mean cycle period (s)')
ax.set_ylabel('duty cycle')
ax.set_title('Each dot = one 4-minute pressure recording')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Train a simple classifier

We do an 75/25 train/test split (stratified so each side gets equal classes) and fit a **logistic regression** — the simplest classifier that still gives a probabilistic output.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1, stratify=y
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(f'Test accuracy: {accuracy_score(y_test, y_pred):.3f}')
print()
print(classification_report(y_test, y_pred, target_names=['healthy', 'leak']))

### Confusion matrix

Where did the classifier get confused? The two off-diagonal cells tell us.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)
ax.set_xticks([0, 1]); ax.set_xticklabels(['pred healthy', 'pred leak'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['true healthy', 'true leak'])
ax.set_title('Confusion matrix')
plt.tight_layout(); plt.show()

## 9. Visualise the decision boundary

Refit on just the two visualised features so we can plot the 2-D decision surface.

In [ ]:
clf2d = LogisticRegression(max_iter=1000).fit(X_train[:, :2], y_train)

xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 2, X[:, 0].max() + 2, 200),
    np.linspace(X[:, 1].min() - 0.05, X[:, 1].max() + 0.05, 200),
)
Z = clf2d.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
cs = ax.contourf(xx, yy, Z, levels=20, cmap='RdBu_r', alpha=0.6)
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
ax.scatter(X[y == 0, 0], X[y == 0, 1], edgecolors='k', label='healthy')
ax.scatter(X[y == 1, 0], X[y == 1, 1], edgecolors='k', label='leak', color='firebrick')
ax.set_xlabel('mean cycle period (s)')
ax.set_ylabel('duty cycle')
ax.set_title('Logistic regression — P(leak)')
plt.colorbar(cs, ax=ax, label='P(leak)')
ax.legend(); plt.tight_layout(); plt.show()

## 10. A more interpretable alternative — decision tree

Logistic regression gives a smooth probability. A **decision tree** gives a set of human-readable rules — useful when a maintenance engineer asks *"what threshold did the model use?"*

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print(f'Tree test accuracy: {accuracy_score(y_test, tree.predict(X_test)):.3f}')

fig, ax = plt.subplots(figsize=(11, 5))
plot_tree(tree, feature_names=FEATURE_NAMES, class_names=['healthy', 'leak'],
          filled=True, rounded=True, fontsize=9, ax=ax)
plt.tight_layout(); plt.show()

## 11. Predict on a brand-new recording

Imagine a maintenance laptop plugged into a coach. It records 4 minutes of pressure, computes the four features, and asks the model: *healthy or leak?*

In [ ]:
# Pretend we just acquired this recording from coach #C7
t_new, p_new, on_new = simulate_pressure(leak_rate=0.022, rng=np.random.default_rng(999))

feats = extract_features(p_new, on_new)
x_new = np.array([[feats[k] for k in FEATURE_NAMES]])

prob_leak = clf.predict_proba(x_new)[0, 1]
verdict = 'LEAK suspected' if prob_leak > 0.5 else 'healthy'

print('Features:', feats)
print(f'P(leak) = {prob_leak:.2%}  ->  {verdict}')

plt.figure(figsize=(10, 3))
plt.plot(t_new, p_new, lw=1, color='darkorange')
plt.title(f'New recording from coach C7   |   {verdict}   (P(leak)={prob_leak:.0%})')
plt.xlabel('Time (s)'); plt.ylabel('Tank pressure (bar)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 12. Classroom exercises

Now ask students to try these — each one teaches a different ML lesson:

1. **Sensor noise.** Increase `noise_std` to `0.03`. Does accuracy drop? Why?
2. **Less data.** Train on only 20 samples per class. What happens to the confusion matrix? (Bias/variance.)
3. **Worse separability.** Make the leak ranges overlap more (e.g. healthy `0.007–0.020`, leaky `0.015–0.030`). Plot the new feature scatter — can you still draw a clean boundary?
4. **Feature ablation.** Train using only `duty_cycle` (1 feature). Then only `mean_period_s`. Which single feature wins?
5. **Wrong sensor placement.** Multiply all healthy pressures by 0.9 to simulate a mis-calibrated sensor on one fleet. Does the model still generalise?
6. **Compare classifiers.** Swap `LogisticRegression` for `KNeighborsClassifier` or `RandomForestClassifier`. Plot accuracy vs model complexity.

---
*This is a teaching simulation — real fleet data adds temperature drift, intermittent leaks that only show during flushing, and many other complications. The pipeline (simulate → featurise → classify → evaluate) stays the same.*